# 配套实践 13-01：建立统一 Action Expert 接口

本练习不重复训练第 12 章的网络，而是用四个可控 toy Expert 模拟自回归、ACT 风格整块解码、Diffusion 迭代去噪和 Flow 迭代积分。它们都返回相同形状的 Action Chunk 候选和同一组元数据，随后在相同绕障任务上比较模式覆盖、碰撞通过率、平滑度和网络调用次数。数值只用于理解接口，不代表真实算法排名。依赖：NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/13-generative-action-model/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 构造四类候选动作并计算统一指标
import matplotlib.pyplot as plt  # 绘制候选轨迹与接口指标
from matplotlib.patches import Rectangle  # 在每个动作空间子图中绘制障碍
np.random.seed(131)  # 固定各 toy Expert 的候选模式与噪声
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 四种内部过程，返回同一个外部对象

统一结果包含 candidates、valid mask、Context 版本、动作 schema、网络调用次数和方法名称。所有候选形状都是 K×H×Da，本例取 40×24×2。内部生成过程可以完全不同，但后处理和评测不再需要知道模型细节。

In [ ]:
candidate_count = 40  # 设置每个 Expert 生成的候选动作块数量
chunk_length = 24  # 设置每个 Action Chunk 的时间长度
action_dimension = 2  # 使用二维位置表示简化动作空间
phase = np.linspace(0.0, 1.0, chunk_length)  # 建立动作块内部的归一化时间
base_x = phase * 2.0 - 1.0  # 让全部动作从横坐标负一移动到正一
def package_result(name, candidates, network_calls):  # 定义统一 Action Expert 返回对象的构造函数
    valid_mask = np.ones((chunk_length, action_dimension), dtype=bool)  # 标记当前固定长度动作块的全部位置有效
    return {"name": name, "candidates": candidates, "valid_mask": valid_mask, "context_version": 27, "action_schema": "ee_xy_delta_v1", "network_calls": network_calls}  # 返回候选与可追踪元数据
def autoregressive_expert():  # 定义逐时间位置生成动作的 toy 自回归 Expert
    candidates = []  # 准备保存全部自回归候选
    for candidate_index in range(candidate_count):  # 逐个采样候选动作块
        mode = 1.0 if np.random.rand() > 0.5 else -1.0  # 随机选择上绕或下绕模式
        target_y = mode * 0.75 * np.sin(np.pi * phase)  # 建立当前模式的理想纵向轮廓
        generated_y = [0.0]  # 从共同起点开始逐步生成纵向位置
        for time_index in range(1, chunk_length):  # 按时间顺序产生后续动作位置
            correction = 0.68 * (target_y[time_index] - generated_y[-1])  # 根据已生成前缀向目标轮廓修正
            generated_y.append(generated_y[-1] + correction + np.random.normal(scale=0.045))  # 把前缀误差与随机性传到下一位置
        candidates.append(np.stack([base_x, np.array(generated_y)], axis=1))  # 保存当前二维自回归动作块
    return package_result("Autoregressive", np.stack(candidates), chunk_length)  # 记录逐位置生成所需的网络调用次数
def act_style_expert():  # 定义一次解码整块动作的 toy ACT 风格 Expert
    candidates = []  # 准备保存全部整块候选
    for candidate_index in range(candidate_count):  # 逐个采样潜在动作风格
        mode = 1.0 if np.random.rand() > 0.5 else -1.0  # 用潜变量符号选择上绕或下绕
        style_amplitude = 0.75 + np.random.normal(scale=0.07)  # 用潜变量幅值模拟示范风格变化
        y_values = mode * style_amplitude * np.sin(np.pi * phase) + np.random.normal(scale=0.018, size=chunk_length)  # 一次产生完整纵向轨迹
        candidates.append(np.stack([base_x, y_values], axis=1))  # 保存当前二维整块动作
    return package_result("ACT-style", np.stack(candidates), 1)  # 记录整块解码只需一次主干调用
def diffusion_style_expert():  # 定义从噪声迭代去噪的 toy Diffusion Expert
    modes = np.where(np.random.rand(candidate_count) > 0.5, 1.0, -1.0)  # 为全部候选选择条件分布中的动作模式
    target_y = modes[:, None] * 0.75 * np.sin(np.pi * phase)[None, :]  # 计算各候选需要恢复的目标轮廓
    current_y = np.random.randn(candidate_count, chunk_length)  # 从 Gaussian 噪声初始化整批动作
    diffusion_calls = 20  # 设置 toy 去噪的网络调用次数
    for denoise_index in range(diffusion_calls):  # 反复把带噪轨迹推向条件动作分布
        progress = (denoise_index + 1) / diffusion_calls  # 计算当前去噪进度
        current_y += 0.22 * (target_y - current_y)  # 模拟网络给出的朝目标模式去噪方向
        current_y += np.random.normal(scale=0.025 * (1.0 - progress), size=current_y.shape)  # 随进度降低随机扰动
    candidates = np.stack([np.repeat(base_x[None, :], candidate_count, axis=0), current_y], axis=2)  # 组成统一二维候选张量
    return package_result("Diffusion-style", candidates, diffusion_calls)  # 返回带二十次网络调用的结果
def flow_style_expert():  # 定义沿条件速度场积分的 toy Flow Expert
    modes = np.where(np.random.rand(candidate_count) > 0.5, 1.0, -1.0)  # 为全部候选选择上绕或下绕目标模式
    target_y = modes[:, None] * 0.75 * np.sin(np.pi * phase)[None, :]  # 建立条件流需要到达的动作轮廓
    current_y = np.random.randn(candidate_count, chunk_length)  # 从 Gaussian 源分布初始化动作
    flow_calls = 12  # 设置 Euler 积分使用的速度场调用次数
    integration_step = 1.0 / flow_calls  # 计算每次积分的生成时间步长
    for integration_index in range(flow_calls):  # 沿 toy 条件速度场逐步积分
        remaining_time = max(1.0 - integration_index * integration_step, integration_step)  # 计算抵达数据端前剩余生成时间
        velocity = (target_y - current_y) / remaining_time  # 构造能在剩余时间抵达目标模式的理想速度
        current_y += integration_step * velocity  # 使用 Euler 更新整批动作位置
    current_y += np.random.normal(scale=0.025, size=current_y.shape)  # 在数据端加入小幅示范变化
    candidates = np.stack([np.repeat(base_x[None, :], candidate_count, axis=0), current_y], axis=2)  # 组成统一二维候选张量
    return package_result("Flow-style", candidates, flow_calls)  # 返回带十二次速度场调用的结果
expert_results = [autoregressive_expert(), act_style_expert(), diffusion_style_expert(), flow_style_expert()]  # 使用同一 Context 调用四种 toy Expert
for result in expert_results:  # 逐个打印统一结果的关键字段
    print(f"{result['name']:17s} shape={result['candidates'].shape}, schema={result['action_schema']}, context=v{result['context_version']}, NFE={result['network_calls']}")  # 显示候选形状与可追踪元数据

**怎样理解结果：** 四行的候选形状、schema 与 Context 版本完全一致，只有方法名称和 NFE 不同。自回归 toy Expert 按 24 个位置逐步调用，ACT 风格一次解码，Diffusion 与 Flow 分别迭代 20 和 12 次。真实实现还应加入生成耗时、随机种子、模型版本和归一化统计版本。

## 2. 用同一后处理器查看四类候选

统一接口使同一绘图、碰撞检查和执行器可以直接处理全部 Expert。下面每种方法显示 20 条候选，灰色矩形是局部过滤器已知的障碍区域。

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), sharex=True, sharey=True)  # 为四种 Expert 创建统一动作空间坐标轴
for axis, result in zip(axes, expert_results):  # 依次绘制每个统一结果对象
    for candidate in result["candidates"][:20]:  # 绘制前二十条候选避免图像过密
        mode_color = "#2563eb" if candidate[chunk_length // 2, 1] > 0 else "#7c3aed"  # 根据中点判断上绕或下绕颜色
        axis.plot(candidate[:, 0], candidate[:, 1], color=mode_color, alpha=0.24)  # 显示完整 Action Chunk 轨迹
    axis.add_patch(Rectangle((-0.25, -0.25), 0.5, 0.5, facecolor="#cbd5e1", edgecolor="#475569"))  # 在相同位置绘制局部已知障碍
    axis.set(title=f"{result['name']}\nNFE={result['network_calls']}", xlabel="x", xlim=(-1.1, 1.1), ylim=(-1.0, 1.0))  # 标注 Expert 名称与网络调用次数
    axis.set_aspect("equal")  # 保持二维动作空间比例一致
axes[0].set_ylabel("y")  # 为共用纵轴标记第二动作维度
fig.suptitle("One postprocessor can inspect candidates from every Expert")  # 强调统一外部接口的作用
fig.tight_layout()  # 调整四幅候选图间距
plt.show()  # 显示四种内部生成过程的统一输出

**怎样理解结果：** 四种 toy Expert 都覆盖上绕和下绕，且可以由同一个障碍过滤器处理。自回归轨迹更容易出现逐步噪声，ACT 风格整块轨迹较紧，迭代生成的差别主要体现在路径形成方式和 NFE。这里人为设置了相近质量，因此不能据此判断真实算法优劣。

## 3. 同时报告质量与生成成本

我们用三个与接口无关的指标评估候选：是否避开障碍、是否同时覆盖上下两个模式，以及二阶差分绝对值表示的轨迹弯折。再把 NFE 放在同一结果中，避免只讨论样本图形。

In [ ]:
def compute_metrics(result):  # 定义统一候选质量与成本指标
    candidates = result["candidates"]  # 从结果对象读取候选动作张量
    inside_x = np.abs(candidates[:, :, 0]) <= 0.25  # 判断每个动作位置是否进入障碍横向范围
    inside_y = np.abs(candidates[:, :, 1]) <= 0.25  # 判断每个动作位置是否进入障碍纵向范围
    collision_free_rate = 1.0 - np.any(inside_x & inside_y, axis=1).mean()  # 计算整条轨迹没有进入障碍的比例
    midpoint_signs = np.sign(candidates[:, chunk_length // 2, 1])  # 使用动作块中点区分上下模式
    mode_coverage = len(np.unique(midpoint_signs[midpoint_signs != 0])) / 2.0  # 计算两个有效模式被覆盖的比例
    second_difference = np.diff(candidates, n=2, axis=1)  # 计算沿动作时间的二阶差分
    roughness = np.mean(np.linalg.norm(second_difference, axis=2))  # 汇总候选轨迹的平均弯折程度
    return collision_free_rate, mode_coverage, roughness, result["network_calls"]  # 返回三个质量指标和生成调用成本
metric_rows = np.array([compute_metrics(result) for result in expert_results])  # 对四种 Expert 应用完全相同的评测
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))  # 创建通过率、覆盖率、平滑度和 NFE 四幅柱状图
metric_names = ["Collision-free rate", "Mode coverage", "Roughness (lower)", "Network calls"]  # 定义四个指标标题
metric_colors = ["#16a34a", "#2563eb", "#ea580c", "#7c3aed"]  # 为四个指标分配颜色
expert_names = [result["name"].replace("-style", "") for result in expert_results]  # 建立紧凑的横轴方法名称
for metric_index, axis in enumerate(axes):  # 依次绘制四个可比较指标
    bars = axis.bar(expert_names, metric_rows[:, metric_index], color=metric_colors[metric_index])  # 绘制四种 Expert 的当前指标
    axis.set_title(metric_names[metric_index])  # 标注当前柱状图所表示的指标
    axis.tick_params(axis="x", rotation=35)  # 旋转方法名称避免文字重叠
    for bar, value in zip(bars, metric_rows[:, metric_index]):  # 逐个读取柱子和指标值
        axis.text(bar.get_x() + bar.get_width() / 2, value + max(metric_rows[:, metric_index].max() * 0.03, 0.01), f"{value:.2f}", ha="center", fontsize=8)  # 在柱子上方写出数值
fig.suptitle("Action quality and generation cost must be reported together")  # 强调质量指标不能脱离实时成本
fig.tight_layout()  # 调整四幅柱状图间距
plt.show()  # 显示统一 Action Expert 评测结果

**怎样理解结果：** 本例四种方法都能覆盖两个模式并大多通过局部障碍检查，但自回归逐步噪声带来更高 roughness；ACT 风格只需一次主干调用，两个迭代方法使用不同 NFE。真实比较还要加入墙钟延迟、闭环成功率、显存与尾延迟，且不能把这个人为 toy 结果当作算法结论。

**本练习的结论：** 统一接口的价值不是让内部方法变得相同，而是让同一动作 schema、后处理、约束和评测能够复用，同时保留方法特有的成本与元数据。